# NEU-ESC Adaptive Interaction
GPU workflow with resumable 4 variants × 5 paired seeds.
Runs marked `smoke` are non-reportable; test metrics are read only after validation checkpoint selection.

In [ ]:
!nvidia-smi
import torch, platform
print(platform.platform(), torch.__version__, torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU accelerator is required for reportable runs'


In [ ]:
# Add the repository as a Kaggle Dataset or clone the pinned public commit.
!git clone https://github.com/AI-Peak/neu-esc-reproduction.git /kaggle/working/NEU_ESC
%cd /kaggle/working/NEU_ESC
!pip install -q 'transformers<5' sentencepiece
!python src_baseline/neu_esc_kaggle.py fetch --data-dir data/raw/kaggle
!python -m src_adaptive.data_audit data/NEU --out reports/adaptive_interaction/kaggle_data_audit.json


In [ ]:
from pathlib import Path
import json, subprocess
variants=['baseline','independent','ungated','adaptive']
seeds=[42,43,44,45,46]
root=Path('runs/adaptive_interaction')
for variant in variants:
  for seed in seeds:
    run=root/f'{variant}-s{seed}'
    if (run/'metrics.json').exists():
      print('skip completed', run); continue
    subprocess.run(['python','src_adaptive/run_experiment.py','--variant',variant,'--seed',str(seed),'--data-dir','data/raw/kaggle','--processed','--revision','86cd7fd4c148980922ac11a2cf5e257f2ba639e1','--out-dir',str(root)],check=True)


In [ ]:
!python src_adaptive/aggregate.py runs/adaptive_interaction --out reports/adaptive_interaction
!python src_adaptive/test_adaptive.py
